# 15. End-to-End Decision Pipeline Demonstration

This notebook demonstrates the complete end-to-end SupportAI production pipeline locally without starting the FastAPI server. It covers:
1. **Configuration Loading**: Loading default configurations.
2. **Preprocessing**: Normalizing input support ticket text.
3. **Classification**: Running predictions through the optimized DistilBERT classifier.
4. **Calibration**: Scaling logits using temperature scaling ($T = 1.1939$) to get calibrated probabilities.
5. **Decision Routing**: Categorizing predictions based on confidence thresholds into:
   - **Tier 1**: Auto-route directly using classifier (confidence $\ge 0.90$)
   - **Tier 2**: Retrieve resolved tickets via FAISS and generate LLM draft response (confidence $0.30 - 0.90$)
   - **Tier 3**: Escalation to human agent (confidence $< 0.30$)
6. **Semantic RAG Retrieval**: Index searching historical resolved banking cases using FAISS index.
7. **Explainability**: Running word attributions via LIME to inspect model attention.

## 1. Setup & Environment Initialization

In [1]:
import os
import sys
import json
from pathlib import Path

# Ensure repository root is in system path
REPO_ROOT = Path("").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["TESTING"] = "true"

from src.models.transformer.decision_engine import DecisionEngine
from src.evaluation.explainability import TicketExplainer
from src.utils.constants import OUTPUT_DIR

## 2. Load the Decision Engine

We load the `DecisionEngine` using a configuration tailored to show all three routing tiers (high-confidence auto-routing, mid-confidence retrieval/LLM fallback, and low-confidence human escalation).

In [2]:
config = {
    "decision_engine": {
        "high_confidence_threshold": 0.90,
        "low_confidence_threshold": 0.30
    },
    "llm": {
        "enabled": True,
        "provider": "huggingface",
        "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
        "max_new_tokens": 20,
        "temperature": 0.0
    }
}

model_dir = OUTPUT_DIR / "models" / "best_model"
retriever_index_dir = OUTPUT_DIR / "retrieval_index"

engine = DecisionEngine(
    model_dir=model_dir,
    retriever_index_dir=retriever_index_dir,
    config=config
)

[07/13/26 21:55:47] INFO     Loading intent classifier from:                                                       
                             C:\Users\gunav\Downloads\SupportAI\outputs\models\best_model

                    INFO     Loaded calibrated temperature scaling: T = 1.1939

                    INFO     Loading semantic retriever from:                                                      
                             C:\Users\gunav\Downloads\SupportAI\outputs\retrieval_index

                    INFO     Loading sentence embedding model: all-MiniLM-L6-v2

                    INFO     Load pretrained SentenceTransformer: all-MiniLM-L6-v2

[07/13/26 21:55:52] INFO     FAISS index loaded successfully from                                                  
                             C:\Users\gunav\Downloads\SupportAI\outputs\retrieval_index

                    INFO     Loading Hugging Face LLM backend: Qwen/Qwen2.5-0.5B-Instruct

[07/13/26 21:55:54] INFO     Using torch_dtype=torch.float16 for LLM backend

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


## 3. Walkthrough of Routing Decisions

Let's pass three representative customer support queries and analyze how they get routed:

In [3]:
# Tier 1: High confidence auto-routing
ticket_1 = "I forgot my passcode and cannot login"
res_1 = engine.route_ticket(ticket_1)
print("=== Ticket 1 (Auto-route) ===")
print(f"Text: {ticket_1}")
print(json.dumps(res_1, indent=2))

print("\n")

# Tier 2: Mid-confidence LLM Fallback (includes FAISS retrieval context + LLM reply)
ticket_2 = "reset card pin number"
res_2 = engine.route_ticket(ticket_2)
print("=== Ticket 2 (LLM Fallback) ===")
print(f"Text: {ticket_2}")
print(json.dumps(res_2, indent=2))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Ticket 1 (Auto-route) ===
Text: I forgot my passcode and cannot login
{
  "intent": "passcode_forgotten",
  "confidence": 0.9707167744636536,
  "route": "classifier",
  "retrieved_docs": [],
  "llm_used": false,
  "reply": "Automated routing to category: passcode_forgotten"
}


=== Ticket 2 (LLM Fallback) ===
Text: reset card pin number
{
  "intent": "get_physical_card",
  "confidence": 0.42254090309143066,
  "route": "fallback",
  "retrieved_docs": [
    {
      "rank": 1,
      "index": 1131,
      "score": 0.8320213556289673,
      "text": "how do i reset my pin, i can't seem to use my card?"
    },
    {
      "rank": 2,
      "index": 1284,
      "score": 0.7538939118385315,
      "text": "do you know how to change my card pin?"
    },
    {
      "rank": 3,
      "index": 735,
      "score": 0.7117278575897217,
      "text": "what do i do with my card pin?"
    }
  ],
  "llm_used": true,
  "reply": "I'm sorry, but I don't have enough information to provide a specific solution

## 4. Ticket Explainability (LIME attributions)

Now let's compute local attributions for the high-confidence prediction to understand which keywords heavily contributed to the predicted intent class.

In [4]:
explainer = TicketExplainer(model_dir=model_dir)

print(f"Explaining text: {ticket_1}")
explanation = explainer.explain_ticket(ticket_1, num_features=5, num_samples=50)

print("\n=== Prediction Summary ===")
print("Predicted Class:", explanation["predicted_class"])
print("Probability:", explanation["predicted_probability"])

print("\n=== Word Attributions ===")
for word, weight in explanation["attributions"]:
    print(f"  {word:<15} : {weight:+.4f}")

[07/13/26 21:56:11] INFO     Loading model and tokenizer for explainability from                                   
                             C:\Users\gunav\Downloads\SupportAI\outputs\models\best_model

Explaining text: I forgot my passcode and cannot login

=== Prediction Summary ===
Predicted Class: passcode_forgotten
Probability: 0.9926396608352661

=== Word Attributions ===
  passcode        : +0.6076
  cannot          : +0.0786
  login           : +0.0759
  forgot          : -0.0709
  my              : -0.0447
